<a href="https://colab.research.google.com/github/sirisha142007/Prompt-Engineering-AI-Productivity/blob/main/AI_Agent_Tools_and_Automation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

!pip install -q gradio requests


In [2]:

import gradio as gr
import requests
import json
import ast
import operator
from datetime import datetime


In [3]:

# Agent memory
conversation_history = []

# Agent state
agent_state = {
    "task": None,
    "action": None,
    "tool_result": None,
    "status": "Ready"
}

print("Agent memory and state initialized!")


Agent memory and state initialized!


In [4]:

# TOOL 1: Calculator

def calculator(expression):

    allowed_operators = {
        ast.Add: operator.add,
        ast.Sub: operator.sub,
        ast.Mult: operator.mul,
        ast.Div: operator.truediv,
        ast.Pow: operator.pow,
        ast.USub: operator.neg,
        ast.UAdd: operator.pos
    }

    def calculate(node):

        if isinstance(node, ast.Constant) and type(node.value) in (int, float):
            return node.value

        if isinstance(node, ast.BinOp) and type(node.op) in allowed_operators:
            if isinstance(node.op, ast.Pow) and abs(calculate(node.right)) > 10:
                raise ValueError("Exponent too large")

            return allowed_operators[type(node.op)](
                calculate(node.left),
                calculate(node.right)
            )

        if isinstance(node, ast.UnaryOp) and type(node.op) in allowed_operators:
            return allowed_operators[type(node.op)](
                calculate(node.operand)
            )

        raise ValueError("Unsupported calculation")

    tree = ast.parse(expression, mode="eval")
    return calculate(tree.body)


# TOOL 2: Date and Time

def get_datetime():

    return datetime.now().strftime("%d-%m-%Y %I:%M:%S %p")


# TOOL 3: Wikipedia Search

def wikipedia_search(query):

    url = "https://en.wikipedia.org/w/api.php"

    params = {
        "action": "query",
        "list": "search",
        "srsearch": query,
        "format": "json"
    }

    response = requests.get(
        url,
        params=params,
        headers={"User-Agent": "ColabAIAgent/1.0"},
        timeout=15
    )

    response.raise_for_status()

    results = response.json()["query"]["search"]

    if not results:
        return "No information found."

    return results[0]["snippet"].replace(
        '<span class="searchmatch">', ""
    ).replace("</span>", "")


# TOOL 4: Text Analyzer

def text_analyzer(text):

    return {
        "Words": len(text.split()),
        "Characters": len(text),
        "Sentences": len([
            s for s in text.split(".") if s.strip()
        ])
    }

print("All four tools created successfully!")


All four tools created successfully!


In [5]:

def decide_action(task):

    text = task.lower()

    if any(word in text for word in [
        "calculate", "plus", "minus", "multiply",
        "divide", "what is", "sum of"
    ]) and any(char.isdigit() for char in text):

        return "calculator"

    elif any(word in text for word in [
        "time", "date", "today"
    ]):

        return "datetime"

    elif any(word in text for word in [
        "search", "who is", "what is", "tell me about",
        "information about"
    ]):

        return "wikipedia"

    elif any(word in text for word in [
        "analyze", "count words", "count characters"
    ]):

        return "text_analyzer"

    else:
        return "general"


In [6]:

def execute_tool(action, task):

    if action == "calculator":

        expression = task.lower()

        expression = expression.replace("calculate", "")
        expression = expression.replace("what is", "")
        expression = expression.replace("plus", "+")
        expression = expression.replace("minus", "-")
        expression = expression.replace("multiplied by", "*")
        expression = expression.replace("multiply", "*")
        expression = expression.replace("divided by", "/")
        expression = expression.replace("divide", "/")
        expression = expression.strip().rstrip("?")

        return calculator(expression)

    elif action == "datetime":

        return get_datetime()

    elif action == "wikipedia":

        query = task

        for phrase in [
            "search for", "search",
            "who is", "what is", "tell me about",
            "information about"
        ]:
            query = query.lower().replace(phrase, "").strip()

        return wikipedia_search(query)

    elif action == "text_analyzer":

        text = task.split(":", 1)[-1].strip()

        if text == task:
            return "Please provide text after a colon."

        return text_analyzer(text)

    else:

        return "I can help with calculations, date and time, Wikipedia searches, and text analysis."


In [7]:

def run_agent(task):

    global agent_state, conversation_history

    if not task.strip():
        return "Please enter a task."

    # Step 1: Understand task
    agent_state["task"] = task
    agent_state["status"] = "Understanding task"

    # Step 2: Decide action
    action = decide_action(task)
    agent_state["action"] = action

    # Step 3: Use tool
    agent_state["status"] = "Executing tool"

    try:
        result = execute_tool(action, task)

        # Step 4: Process result
        agent_state["tool_result"] = result

        # Step 5: Generate response
        answer = f"""
        AI AGENT RESPONSE

        Your Task:
        {task}

        Selected Action:
        {action}

        Tool Result:
        {result}

        Status: Completed
        """

        conversation_history.append({
            "user": task,
            "assistant": answer
        })

        agent_state["status"] = "Completed"

        return answer

    except Exception as e:

        agent_state["status"] = "Error"

        return f"Tool execution failed: {str(e)}"


In [8]:

print(run_agent("Calculate 25 plus 35"))

print(run_agent("What is the current date and time?"))

print(run_agent("Who is Albert Einstein?"))



        AI AGENT RESPONSE

        Your Task:
        Calculate 25 plus 35

        Selected Action:
        calculator

        Tool Result:
        60

        Status: Completed
        

        AI AGENT RESPONSE

        Your Task:
        What is the current date and time?

        Selected Action:
        datetime

        Tool Result:
        27-11-2026 12:49:03 PM

        Status: Completed
        

        AI AGENT RESPONSE

        Your Task:
        Who is Albert Einstein?

        Selected Action:
        wikipedia

        Tool Result:
        Albert Einstein (14 March 1879 – 18 April 1955) was a German-born theoretical physicist best known for developing the theory of relativity. Einstein also

        Status: Completed
        


In [9]:

with gr.Blocks(
    theme=gr.themes.Soft(),
    title="AI Agent Automation"
) as demo:

    gr.Markdown(
        """
        # 🤖 AI Agent – Tools & Automation

        ### Your personal AI task assistant

        Enter a task and let the agent decide which tool to use.
        """
    )

    user_input = gr.Textbox(
        label="Enter Your Task",
        placeholder="Example: Calculate 100 plus 250",
        lines=2
    )

    run_button = gr.Button(
        "Run AI Agent",
        variant="primary"
    )

    output = gr.Textbox(
        label="Agent Response",
        lines=9
    )

    run_button.click(
        fn=run_agent,
        inputs=user_input,
        outputs=output
    )

    gr.Markdown(
        """
        ### Available Tools
        - Calculator
        - Date and Time
        - Wikipedia Search
        - Text Analyzer
        """
    )

demo.launch(share=True)


/tmp/ipykernel_3813/525012639.py:1: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://4d4b9fc4d15e17464f.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [10]:

print("AGENT CONVERSATION HISTORY")

for chat in conversation_history:

    print("\nUser:", chat["user"])
    print("Agent:", chat["assistant"])

print("\nCURRENT AGENT STATE")

print(json.dumps(agent_state, indent=4, default=str))


AGENT CONVERSATION HISTORY

User: Calculate 25 plus 35
Agent: 
        AI AGENT RESPONSE

        Your Task:
        Calculate 25 plus 35

        Selected Action:
        calculator

        Tool Result:
        60

        Status: Completed
        

User: What is the current date and time?
Agent: 
        AI AGENT RESPONSE

        Your Task:
        What is the current date and time?

        Selected Action:
        datetime

        Tool Result:
        27-11-2026 12:49:03 PM

        Status: Completed
        

User: Who is Albert Einstein?
Agent: 
        AI AGENT RESPONSE

        Your Task:
        Who is Albert Einstein?

        Selected Action:
        wikipedia

        Tool Result:
        Albert Einstein (14 March 1879 – 18 April 1955) was a German-born theoretical physicist best known for developing the theory of relativity. Einstein also

        Status: Completed
        

CURRENT AGENT STATE
{
    "task": "Who is Albert Einstein?",
    "action": "wikipedia",
    "to